# Background:
- Some sessions have bad correction results when calculated by post hoc z-drift calculation.
- The same sessions show OK-ish estimation from online motion estimation.
- Check which is more reliable using napari.

In [1]:
import napari
from pathlib import Path
from tifffile import imread
import h5py
import numpy as np

In [13]:
ZSTACK_REG_TAG = {
    0: '0_reg_ch_2',
    1: '0_reg_ch_1',
    2: '1_reg_ch_2',
    3: '1_reg_ch_1',
    4: '2_reg_ch_2',
    5: '2_reg_ch_1',
    6: '3_reg_ch_2',
    7: '3_reg_ch_1',
}

In [30]:
# session_key = '790322_2025-08-21'
session_key = '785054_2025-07-18'
plane_ind = 1
plane_id = f'VISp_{plane_ind}'
zstack_reg_tag = ZSTACK_REG_TAG[plane_ind]

zstack_fn_list = list(Path(r'D:\online motion correction').glob(f'{session_key}\\local_zstacks\\*{zstack_reg_tag}.tif'))
assert len(zstack_fn_list) == 1, f'Expected one zstack file, found {len(zstack_fn_list)}'
zstack_fn = zstack_fn_list[0]
zstack = imread(zstack_fn)
h5_fn = Path(r'D:\online motion correction') / session_key / 'onemin_emf' / f'{plane_id}_onemin_emf.h5'
with h5py.File(h5_fn, 'r') as f:
    emf = f['data'][:]
numpy_fn = Path(r'D:\online motion correction') / session_key / 'zdrift_results' / f'{plane_id}_zdrift_results.npy'
zdrift = np.load(numpy_fn, allow_pickle=True).item()


In [21]:
zdrift['matched_plane_indices']

array([47, 49, 49, 48, 48, 48, 48, 47, 48, 48, 48, 47, 47, 46, 48, 45, 47,
       40, 47, 46, 46, 46, 40, 48, 46, 46, 47, 47, 45, 43, 46, 46, 45, 45,
       45, 46, 47, 47, 46, 46, 45, 47, 47, 46, 47, 46, 46, 46, 47, 47, 47,
       45, 47, 46, 46, 42, 47, 42, 47, 46, 46, 46, 46], dtype=int64)

In [31]:
viewer = napari.Viewer()
viewer.add_image(emf, name=f'{session_key} {plane_id} EMF')
viewer.add_image(zstack, name=f'{session_key} {plane_id} zstack')

<Image layer '785054_2025-07-18 VISp_1 zstack' at 0x1cedd195a90>

In [28]:
viewer = napari.Viewer()
viewer.add_image(zstack, name='zstack', colormap='gray')
mpi = zdrift['matched_plane_indices']
# for i in range(0, len(emf), 11):
#     viewer.add_image(emf[i], name=f'emf_{i:02} matched {mpi[i]}', colormap='gray')
i = np.argmax(mpi)
viewer.add_image(emf[i], name=f'emf_{i:02} matched {mpi[i]}', colormap='gray')
i = np.argmin(mpi)
viewer.add_image(emf[i], name=f'emf_{i:02} matched {mpi[i]}', colormap='gray')

<Image layer 'emf_17 matched 40' at 0x1cefa29b190>